![image](car.jpeg)

**Car-ing is sharing**, an auto dealership company for car sales and rental, is taking their services to the next level thanks to **Large Language Models (LLMs)**.

As their newly recruited AI and NLP developer, you've been asked to prototype a chatbot app with multiple functionalities that not only assist customers but also provide support to human agents in the company.

The solution should receive textual prompts and use a variety of pre-trained Hugging Face LLMs to respond to a series of tasks, e.g. classifying the sentiment in a car’s text review, answering a customer question, summarizing or translating text, etc.


In [14]:
# Import necessary packages
import pandas as pd
import torch

from transformers import logging
logging.set_verbosity(logging.WARNING)

reviews = pd.read_csv("data/car_reviews.csv", sep=";", encoding="utf-8-sig")
print(reviews.head())

                                              Review     Class
0  I am very satisfied with my 2014 Nissan NV SL....  POSITIVE
1  The car is fine. It's a bit loud and not very ...  NEGATIVE
2  My first foreign car. Love it, I would buy ano...  POSITIVE
3  I've come across numerous reviews praising the...  NEGATIVE
4  I've been dreaming of owning an SUV for quite ...  POSITIVE


In [15]:
# Start your code here!
# Import necessary packages
import pandas as pd
import torch

from transformers import logging
logging.set_verbosity(logging.WARNING)

import evaluate
from transformers import pipeline

# Load the dataset
reviews = pd.read_csv("data/car_reviews.csv", sep=";", encoding="utf-8-sig")
print(reviews.head())

# Reference labels: map text classes to 1/0
references = [1 if label == "POSITIVE" else 0 for label in reviews["Class"]]

# Load the sentiment model
sentiment_model = pipeline("text-classification", model="distilbert-base-uncased-finetuned-sst-2-english")

# Run the model on the reviews
predicted_labels = sentiment_model(reviews["Review"].tolist(), truncation=True)
print(predicted_labels)

# Extract the labels and map them to {0,1}
predictions = [1 if output["label"] == "POSITIVE" else 0 for output in predicted_labels]

# Calculate the metrics
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

accuracy_result = accuracy.compute(references=references, predictions=predictions)["accuracy"]
f1_result = f1.compute(references=references, predictions=predictions)["f1"]

print(f"Accuracy: {accuracy_result}")
print(f"F1: {f1_result}")


from transformers import pipeline
import evaluate

# Extract the first two sentences of the first review
first_review = reviews["Review"].iloc[0]
first_two_sentences = ". ".join(first_review.split(". ")[:2]) + "."
print(first_two_sentences)

# Load the English-to-Spanish translation model
translator = pipeline(task="translation", model="Helsinki-NLP/opus-mt-en-es")

# Translate and store the text
translated_review = translator(first_two_sentences, max_length=40)[0]["translation_text"]
print(translated_review)

# Load the reference translations
with open("data/reference_translations.txt", "r") as file:
    lines = file.readlines()
references = [line.strip() for line in lines]
print(references)

# Calculate the BLEU score
bleu = evaluate.load("bleu")
bleu_score = bleu.compute(predictions=[translated_review], references=[references])
print(bleu_score)
print(f"BLEU: {bleu_score['bleu']}")






from transformers import pipeline

# Inputs: the question and the 2nd review (index 1) as context
context = reviews["Review"].iloc[1]
question = "What did he like about the brand?"

# Load the extractive QA model
qa_model = pipeline(task="question-answering", model="deepset/minilm-uncased-squad2")

# Run the model and extract the text answer
outputs = qa_model(question=question, context=context)
print(outputs)

answer = outputs["answer"]
print(answer)

Device set to use cpu


                                              Review     Class
0  I am very satisfied with my 2014 Nissan NV SL....  POSITIVE
1  The car is fine. It's a bit loud and not very ...  NEGATIVE
2  My first foreign car. Love it, I would buy ano...  POSITIVE
3  I've come across numerous reviews praising the...  NEGATIVE
4  I've been dreaming of owning an SUV for quite ...  POSITIVE
[{'label': 'POSITIVE', 'score': 0.929397702217102}, {'label': 'POSITIVE', 'score': 0.8654273152351379}, {'label': 'POSITIVE', 'score': 0.9994640946388245}, {'label': 'NEGATIVE', 'score': 0.9935314059257507}, {'label': 'POSITIVE', 'score': 0.9986565113067627}]
Accuracy: 0.8
F1: 0.8571428571428571
I am very satisfied with my 2014 Nissan NV SL. I use this van for my business deliveries and personal use.


Device set to use cpu


Estoy muy satisfecho con mi Nissan NV SL 2014. Uso esta camioneta para mis entregas de negocios y uso personal.
['Estoy muy satisfecho con mi Nissan NV SL 2014. Utilizo esta camioneta para mis entregas comerciales y uso personal.', 'Estoy muy satisfecho con mi Nissan NV SL 2014. Uso esta furgoneta para mis entregas comerciales y uso personal.']


Some weights of the model checkpoint at deepset/minilm-uncased-squad2 were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


{'bleu': 0.7794483794144497, 'precisions': [0.9090909090909091, 0.8571428571428571, 0.75, 0.631578947368421], 'brevity_penalty': 1.0, 'length_ratio': 1.0476190476190477, 'translation_length': 22, 'reference_length': 21}
BLEU: 0.7794483794144497
{'score': 0.477361261844635, 'start': 569, 'end': 594, 'answer': 'ride quality, reliability'}
ride quality, reliability


In [16]:
from transformers import pipeline
import evaluate

# The last review in the dataset
last_review = reviews["Review"].iloc[-1]

# Load a summarization model
summarizer = pipeline(task="summarization", model="facebook/bart-large-cnn")

# Summarize to roughly 50-55 tokens
outputs = summarizer(last_review, min_length=50, max_length=55, truncation=True)
summarized_text = outputs[0]["summary_text"]
print(summarized_text)

# Bias analysis: maximum toxicity and regard
toxicity = evaluate.load("toxicity", module_type="measurement")
toxicity_result = toxicity.compute(predictions=[summarized_text], aggregation="maximum")
print(toxicity_result)

regard = evaluate.load("regard", module_type="measurement")
regard_result = regard.compute(data=[summarized_text], aggregation="maximum")
print(regard_result)

Device set to use cpu
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


The Nissan Rogue provides me with the desired SUV experience without burdening me with an exorbitant payment. Handling and styling are great; I have hauled 12 bags of mulch in the back with the seats down and could have held more. The engine delivers strong


Device set to use cpu


{'max_toxicity': 0.00013863427739124745}


Device set to use cpu


{'max_regard': {'positive': 0.6263338923454285, 'neutral': 0.20273476839065552, 'other': 0.12291575223207474, 'negative': 0.04801557958126068}}
